# 06 · 고정한 경보 기준은 어떤 오류와 불확실성을 만드는가

04에서 선택한 Cat28의 **검증 음성 점수**로 경보 임계값을 정한다. 그 임계값을 시험 산모에게 그대로 적용하고, 구간·기록·유지된 정상 시간 단위의 성능을 산모 bootstrap으로 평가한다. 검증 집합에서 확률 보정도 적합하되, 시험 결과로 기준을 다시 선택하지 않는다.

**흐름:** 저장 예측 읽기 → 임계값 예시와 고정 → 운영 지표·신뢰구간 → 확률 보정 → TP/TN/FP/FN 사례 집계 → 사례 SHAP 저장·훈련 SHAP 집계 → 저장

**입력:** `data/results/`의 `pred_cat28.csv`, `protocol.json`; 사례·SHAP에는 `cnn_signals.npy`, `features.csv`, `model_selection.json`, 선택된 `models/cat28_*.cbm`도 사용한다.  
**출력:** 임계값, 원래·보정 확률의 시험 예측, 운영 지표·보정 차이, 사례·SHAP 표와 실행 상태

원본 `03-요약된-파일/06-경보-기준과-불확실성.ipynb`의 선택·평가 규칙을 유지한다. 코호트와 모델은 이 폴더의 00→01→02→04 결과를 따른다. 사례 파형이나 모델 파일이 없으면 해당 단계를 보류하고, 계산 가능한 운영 지표는 남긴다.

In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.linear_model import LogisticRegression

RESULTS = Path("data/results")
N_BOOTSTRAP = 2000
BOOTSTRAP_SEED = 42
CALIBRATION_SEED = 42
ID_TYPES = {key: str for key in ("row_id", "record_id", "mother_id", "site")}
PREDICTION_COLUMNS = [
    "row_id", "record_id", "mother_id", "site", "target", "split",
    "record_all_normal", "signal_index", "score", "threshold",
]
METRICS = (
    "AP", "AUROC", "Brier", "sensitivity", "specificity", "PPV",
    "normal_record_alarm", "alarms_per_normal_hour",
)
METRIC_COLUMNS = ["metric", "point", "low", "high", "valid_bootstrap", "mothers", "segments"]

### 출력에 남기는 오류 정보

오류는 고정된 분류와 건수로만 요약한다. `other_error`의 상세 원인은 내부 오류 파일에서 확인한다. 건수는 오류 내역 수이며, 같은 기록이 여러 번 포함될 수 있다.

In [ ]:
def issue_category(reason):
    # Match known phrases, but return ONLY fixed labels: never echo an exception.
    # Example: "FileNotFoundError: /private/DEMO.json" -> "missing_input".
    text = str(reason)
    groups = {
        "missing_input": ("FileNotFoundError", "No such file", "[Errno 2]"),
        "missing_dependency": ("No module named", "cannot import name", "ImportError"),
        "fhr_missing": ("FHR 결측률",),
        "invalid_wave": ("FHR과 TOCO", "측정 간격", "FHR은 0 이상", "신호 길이"),
        "invalid_bbox": ("Bbox",),
        "invalid_id": ("ID가", "산모 ID", "산모 목록", "행 정보 불일치", "구간 정보 불일치"),
        "invalid_labels": ("라벨이 올바르지", "라벨 수", "라벨·점수"),
        "no_usable_data": ("구간이 없습니다", "비었", "남은 산모가 없습니다"),
        "split_unavailable": ("분할", "양성·음성", "정상·비정상", "least populated class", "test_size"),
        "checkpoint_mismatch": ("체크포인트", "CNN 입력·설정"),
        "io_error": ("PermissionError", "Permission denied", "No space left", "OSError"),
        "invalid_value": ("ValueError", "TypeError", "유한", "could not convert", "ufunc"),
    }
    for category, phrases in groups.items():
        if any(phrase in text for phrase in phrases):
            return category
    return "other_error"

In [ ]:
def show_issue_counts(reasons):
    # Count issue entries, not unique people; one record may have several entries.
    categories = pd.Series([issue_category(reason) for reason in reasons], dtype="string")
    counts = categories.value_counts().rename_axis("category").to_frame("count")
    print("제외·오류 내역 수 =", len(categories))
    # Full messages, IDs, paths, and jobs stay in the internal CSV files.
    with pd.option_context("display.max_rows", None, "display.max_columns", None):
        display(counts)

## 1. 선택된 Cat28의 검증·시험 예측을 읽는다

한 행은 유지된 5분 구간 하나다. `score`는 이상 확률, `target`은 전문가의 구간 라벨이다. `record_all_normal`은 제외된 구간까지 포함한 **원래 기록 전체의 라벨**로 00에서 정한 값이다. 남아 있는 정상 구간만 보고 다시 계산하지 않는다.

In [ ]:
predictions = pd.DataFrame()
protocol = {}
input_error = None
issues = []
try:
    # IDs are strings; "00DEMO527183" is a fictional example with leading zeros.
    # 04 saved validation and test windows together; no model fitting happens here.
    predictions = pd.read_csv(RESULTS / "pred_cat28.csv", dtype=ID_TYPES)
    protocol = json.loads((RESULTS / "protocol.json").read_text(encoding="utf-8"))
except (OSError, ValueError) as error:
    input_error = str(error)

In [ ]:
ready = False
validation = pd.DataFrame()
testing = pd.DataFrame()
if input_error is None:
    try:
        predictions[PREDICTION_COLUMNS]
        if predictions.empty or predictions["row_id"].duplicated().any():
            raise ValueError("Cat28 예측이 비었거나 구간 ID가 중복됩니다")
        if not predictions["target"].isin([0, 1]).all() or not np.isfinite(predictions["score"]).all():
            raise ValueError("예측 라벨 또는 점수가 올바르지 않습니다")
        if set(predictions["split"]) != {"val", "test"}:
            raise ValueError("Cat28 예측에는 검증·시험 분할이 모두 필요합니다")
        # Example: mother A must not have windows in both validation and test.
        if predictions.groupby("mother_id")["split"].nunique().max() > 1:
            raise ValueError("같은 산모가 검증과 시험에 함께 있습니다")
        # Validation determines the threshold; test measures its frozen performance.
        validation = predictions.loc[predictions["split"].eq("val")].copy()
        testing = predictions.loc[predictions["split"].eq("test")].copy()
        if set(testing["target"].unique()) != {0, 1}:
            raise ValueError("시험 평가에는 정상·비정상 구간이 모두 필요합니다")
        ready = True
    except (KeyError, ValueError) as error:
        input_error = str(error)
if input_error is not None:
    issues.append(dict(stage="input", job="pred_cat28", reason=input_error))

In [ ]:
print("ready =", ready)
if ready:
    # Windows determine metric numerators; mothers determine bootstrap clusters.
    display(predictions.groupby("split").agg(
        mothers=("mother_id", "nunique"), windows=("row_id", "size"),
        abnormal_fraction=("target", "mean"),
    ))
else:
    print('06 미산출: 04의 예측을 확인하세요.', '[details omitted from notebook output]')

## 2. 검증 특이도 90% 이상인 임계값을 고정한다

검증 음성 점수를 정렬해 `ceil(0.9 × 음성 수)`번째 값을 찾는다. 그 값보다 아주 조금 큰, 다음 표현 가능한 실수를 임계값으로 쓴다. 경보 조건은 `score >= threshold`다. 경계와 같은 음성 점수까지 비경보로 처리하므로 동점이 많으면 검증 특이도가 90%보다 높아질 수 있다.

In [ ]:
def threshold90(labels, scores):
    negative_scores = np.sort(np.asarray(scores)[np.asarray(labels) == 0])
    if not len(negative_scores):
        raise ValueError("검증 음성이 없어 임계값을 정할 수 없습니다")
    # Example: 10 negatives -> ceil(0.9 * 10) = 9th value -> index 8.
    boundary_index = int(np.ceil(0.9 * len(negative_scores))) - 1
    # A high-scoring positive never enters this negative-score order statistic.
    boundary = negative_scores[boundary_index]
    # Move above the boundary because score == threshold triggers an alarm.
    # Example: boundary 0.8 -> threshold 0.8000000000000002.
    return np.nextafter(boundary, np.inf)

In [ ]:
# Ten negatives, with a tie at the 9th value: every score is <= 0.8.
example_negative_scores = np.array([0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.8, 0.8])
example_threshold = threshold90(np.zeros(10), example_negative_scores)
print("threshold with tied negatives =", repr(example_threshold))
print("negative alarms =", (example_negative_scores >= example_threshold).tolist())
print("validation specificity =", np.mean(example_negative_scores < example_threshold))
# Expected specificity: 1.0, rather than exactly 0.9, because the boundary has ties.

In [ ]:
threshold = None
operating_predictions = pd.DataFrame(columns=PREDICTION_COLUMNS + ["alarm"])
if ready:
    try:
        threshold = float(threshold90(validation["target"], validation["score"]))
        operating_predictions = testing.copy()
        # Every test row receives exactly the same validation-derived threshold.
        operating_predictions["threshold"] = threshold
        # Example: score=0.60 and frozen threshold=0.55 -> alarm=True.
        operating_predictions["alarm"] = operating_predictions["score"] >= threshold
    except ValueError as error:
        ready = False
        input_error = str(error)
        issues.append(dict(stage="threshold", job="validation_negatives", reason=input_error))

In [ ]:
if ready:
    validation_negatives = validation.loc[validation["target"].eq(0), "score"]
    print("frozen threshold =", threshold)
    print("validation specificity =", np.mean(validation_negatives < threshold))
    # Rows are expert labels; columns are the frozen model alarms.
    display(pd.crosstab(
        operating_predictions["target"], operating_predictions["alarm"],
        rownames=["Expert"], colnames=["Alarm"],
    ))

## 3. 구간·기록·시간의 분모를 구별한다

AP·AUROC·Brier와 함께 민감도, 특이도, PPV를 계산한다. 정상 기록 경보율은 `원래 모든 라벨이 정상인 기록 중 경보 창이 하나라도 있는 기록 수 / 시험에서 유지된 구간이 있는 그런 정상 기록 수`다.

정상 시간당 경보는 `정상 기록에서 나온 경보 창 수 / (유지된 정상 기록의 창 수 / 12)`다. 5분 창 12개가 1시간이므로 나누는 값이 12다. 연속된 경보를 묶은 episode 수가 아니며, 이상 기록 안에 섞인 정상 창은 이 분모에 들어가지 않는다.

In [ ]:
def prepare_metrics(prediction):
    labels = prediction["target"].to_numpy(dtype=int)
    scores = prediction["score"].to_numpy(dtype=float)
    if not len(prediction) or set(labels) - {0, 1} or not np.isfinite(scores).all():
        raise ValueError("평가 라벨·점수가 올바르지 않거나 평가 집합이 비었습니다")

    # Example: mother IDs [A, A, B] -> mother_index [0, 0, 1].
    mothers, mother_index = np.unique(prediction["mother_id"], return_inverse=True)
    # Example: scores [0.2, 0.9, 0.5] -> descending row order [1, 2, 0].
    order = np.argsort(-scores, kind="stable")
    # Example: sorted scores [0.9, 0.9, 0.3] -> group ends [1, 2].
    # Tied scores are one threshold, independent of their row order.
    ends = np.r_[np.flatnonzero(np.diff(scores[order])), len(prediction) - 1]
    return dict(
        labels=labels, scores=scores, mothers=mothers,
        mother_index=mother_index, order=order, ends=ends,
    )

In [ ]:
def prepare_operating_data(prediction):
    data = prepare_metrics(prediction)
    thresholds = prediction["threshold"].to_numpy(dtype=float)
    if not np.isfinite(thresholds).all():
        raise ValueError("운영 평가에 유한한 임계값이 필요합니다")
    # Recompute from scores and thresholds, independent of the saved alarm column.
    data["alarm"] = data["scores"] >= thresholds
    # A record with an excluded abnormal window still has normal=False here.
    data["normal"] = prediction["record_all_normal"].to_numpy(dtype=bool)

    # Example: records [R1, R1, R2] -> record_index [0, 0, 1], first_rows [0, 2].
    _, first_rows, record_index = np.unique(
        prediction["record_id"], return_index=True, return_inverse=True,
    )
    # Example: two records from mother M1 both use M1's bootstrap count.
    data["record_mother"] = data["mother_index"][first_rows]
    data["normal_record"] = data["normal"][first_rows]
    # One alarming window is enough to mark the whole record as alarming.
    data["record_alarm"] = np.bincount(record_index, weights=data["alarm"]) > 0
    return data

In [ ]:
def rank_metrics(data, counts=None):
    if counts is None:
        # Point estimate: every mother appears once, so every window has weight 1.
        counts = np.ones(len(data["mothers"]))
    # Example: mother counts [2, 0] -> weights [2, 2, 0] for [A, A, B].
    weights = counts[data["mother_index"]]
    labels = data["labels"]
    # Denominators count weighted abnormal and normal *windows*.
    positive = np.sum(weights * labels)
    negative = np.sum(weights * (1 - labels))
    # Skip a resample containing only normal or only abnormal windows.
    if not positive or not negative:
        return None

    order, ends = data["order"], data["ends"]
    # Count windows at or above each distinct score threshold.
    # Example: distinct descending scores with labels [1, 0, 1] -> TP [0, 1, 1, 2].
    true_positive = np.r_[0, np.cumsum((weights * labels)[order])[ends]]
    false_positive = np.r_[0, np.cumsum((weights * (1 - labels))[order])[ends]]
    recall = true_positive / positive
    false_positive_rate = false_positive / negative
    predicted_positive = np.maximum(true_positive[1:] + false_positive[1:], 1e-30)
    # Example: descending labels [1, 0, 1, 0] -> AP = (1 + 2/3)/2 = 5/6.
    ap = np.sum(np.diff(recall) * true_positive[1:] / predicted_positive)
    # Those labels with distinct scores give AUROC=3/4.
    auc = np.sum(np.diff(false_positive_rate) * (recall[1:] + recall[:-1]) / 2)
    # Example: label 1, score 0.9 contributes (0.9 - 1)^2 = 0.01.
    brier = np.sum(weights * (data["scores"] - labels) ** 2) / weights.sum()
    return dict(AP=ap, AUROC=auc, Brier=brier)

In [ ]:
def operating_rates(data, counts):
    weights = counts[data["mother_index"]]
    labels, alarms = data["labels"], data["alarm"]
    positive = np.sum(weights * labels)
    negative = np.sum(weights * (1 - labels))
    true_alarm = np.sum(weights * labels * alarms)
    false_alarm = np.sum(weights * (1 - labels) * alarms)
    # Example: TP=1, FN=1 -> sensitivity=1/2; TP=1, FP=2 -> PPV=1/3.
    # Specificity = 1 - FP/negative windows; no alarms means undefined PPV.
    rates = dict(
        sensitivity=true_alarm / positive,
        specificity=1 - false_alarm / negative,
        PPV=true_alarm / (true_alarm + false_alarm) if true_alarm + false_alarm else np.nan,
    )
    # Each record inherits its mother's draw count; keep only normal records.
    record_weights = counts[data["record_mother"]] * data["normal_record"]
    # Count retained windows from all-normal records, not every target=0 window.
    normal_hours = np.sum(weights * data["normal"]) / 12
    # Example: 1 alarming record / 2 normal records -> 0.5.
    rates["normal_record_alarm"] = (
        np.sum(record_weights * data["record_alarm"]) / record_weights.sum()
        if record_weights.sum() else np.nan
    )
    # Example: 1 alarm in 4 normal windows -> 1 / (4/12) = 3 alarms per hour.
    rates["alarms_per_normal_hour"] = (
        np.sum(weights * data["normal"] * alarms) / normal_hours if normal_hours else np.nan
    )
    return rates

In [ ]:
def weighted_metrics(data, counts=None):
    if counts is None:
        # Point estimate: every mother is counted once.
        counts = np.ones(len(data["mothers"]))
    result = rank_metrics(data, counts)
    if result is None:
        # The original rule skips one-class draws for all eight metrics.
        return None
    result.update(operating_rates(data, counts))
    return result

In [ ]:
# Records A and B are entirely normal; record C contains two abnormal windows.
example_prediction = pd.DataFrame({
    "row_id": [f"window_{i}" for i in range(8)],
    "record_id": ["A", "A", "B", "B", "C", "C", "C", "C"],
    "mother_id": ["M1", "M1", "M2", "M2", "M3", "M3", "M3", "M3"],
    "target": [0, 0, 0, 0, 1, 1, 0, 0],
    "record_all_normal": [1, 1, 1, 1, 0, 0, 0, 0],
    "score": [0.6, 0.2, 0.3, 0.1, 0.8, 0.4, 0.7, 0.2],
    "threshold": 0.5,
})
example_data = prepare_operating_data(example_prediction)
example_metrics = weighted_metrics(example_data)
print("window alarms =", example_data["alarm"].tolist())
print("normal record alarm rate =", example_metrics["normal_record_alarm"])  # 1/2
print("alarms per normal hour =", example_metrics["alarms_per_normal_hour"])  # 3
print("sensitivity, PPV =", example_metrics["sensitivity"], example_metrics["PPV"])

## 4. 같은 산모를 함께 재표집해 95% 구간을 구한다

시험 산모가 M명이면 매번 M명을 복원추출한다. 산모를 두 번 뽑으면 그 산모의 모든 구간과 기록을 두 번 센다. 확률 보정 전후의 차이에서는 양쪽에 **같은 산모 추출 횟수**를 적용한다.

어떤 반복에서 경보가 없으면 PPV가, 정상 기록이 없으면 정상 기록·시간 지표가 결측일 수 있다. 각 지표는 유한한 값만 모으므로 `valid_bootstrap`이 서로 다를 수 있다. 모델·임계값·보정기를 다시 적합하지 않는다.

In [ ]:
def align_rows(left, right):
    # Example: left IDs [A, B], right IDs [B, A] -> reorder right to [A, B].
    # Missing B is an error; comparing only the intersection changes the cohort.
    if left["row_id"].duplicated().any() or right["row_id"].duplicated().any():
        raise ValueError("구간 ID가 중복됩니다")
    if set(left["row_id"]) != set(right["row_id"]):
        raise ValueError("비교할 구간 집합이 다릅니다")
    # Reordering rows never changes their labels, records, or mother IDs.
    aligned = right.set_index("row_id").loc[left["row_id"]].reset_index()
    for column in ("mother_id", "target", "record_id"):
        if not np.array_equal(left[column].to_numpy(), aligned[column].to_numpy()):
            raise ValueError("구간 정보 불일치: " + column)
    return aligned

In [ ]:
def resample_metrics(left_data, right_data, n_boot, seed):
    samples = {metric: [] for metric in METRICS}
    mother_count = len(left_data["mothers"])
    rng = np.random.default_rng(seed)
    for _ in range(n_boot):
        # Example: mother draw [0, 0, 2] -> counts [2, 0, 1].
        draws = rng.integers(mother_count, size=mother_count)
        counts = np.bincount(draws, minlength=mother_count)
        # The same mother counts weight both models, including repeat records and twins.
        left_result = weighted_metrics(left_data, counts)
        right_result = weighted_metrics(right_data, counts) if right_data is not None else None
        if left_result is None or (right_data is not None and right_result is None):
            continue
        for metric in METRICS:
            value = left_result[metric]
            if right_data is not None:
                # Example: calibrated Brier 0.10 - raw Brier 0.12 -> -0.02.
                value -= right_result[metric]
            # Example: a draw with no alarms contributes to specificity, but not PPV.
            if np.isfinite(value):
                samples[metric].append(value)
    return samples

In [ ]:
def metric_intervals(left, right=None, n_boot=N_BOOTSTRAP, seed=BOOTSTRAP_SEED):
    left_data = prepare_operating_data(left)
    right_data = None
    if right is not None:
        right_data = prepare_operating_data(align_rows(left, right))
    point = weighted_metrics(left_data)
    right_point = weighted_metrics(right_data) if right_data is not None else None
    if point is None or (right_data is not None and right_point is None):
        raise ValueError("평가에는 정상·비정상 구간이 모두 필요합니다")
    # Optional right table changes an absolute estimate into a left-minus-right difference.
    if right_point is not None:
        point = {metric: value - right_point[metric] for metric, value in point.items()}
    samples = resample_metrics(left_data, right_data, n_boot, seed)

    rows = []
    for metric, value in point.items():
        low, high = np.nan, np.nan
        if samples[metric]:
            # Central 95% of valid draws: 2.5th and 97.5th percentiles.
            low, high = np.quantile(samples[metric], [0.025, 0.975])
        rows.append(dict(
            metric=metric, point=value, low=low, high=high,
            # Each metric records its own count of usable draws.
            valid_bootstrap=len(samples[metric]),
            mothers=len(left_data["mothers"]), segments=len(left),
        ))
    return pd.DataFrame(rows)

In [ ]:
# An identical table has zero paired differences wherever the metric is defined.
example_difference = metric_intervals(example_prediction, example_prediction, n_boot=20)
display(example_difference[["metric", "point", "low", "high", "valid_bootstrap"]])
# Drawing only record C leaves no normal-record denominator, so valid counts differ.

In [ ]:
performance = pd.DataFrame(columns=METRIC_COLUMNS)
if ready:
    try:
        performance = metric_intervals(operating_predictions)
    except ValueError as error:
        issues.append(dict(stage="operating", job="bootstrap", reason=str(error)))

# One row per metric; all point estimates refer to the same test prediction table.
display(performance)

## 5. 검증 집합에서 확률을 보정한다

`q = sigmoid(a × logit(p) + b)` 형태의 로지스틱 보정기를 검증 라벨과 점수로 적합한다. 원점수 0과 1은 logit이 무한대가 되지 않도록 먼저 제한한다. 보정된 검증 음성 점수로 같은 90% 규칙의 임계값을 정한다.

증가 변환이면 순위가 유지되지만, clipping·동점이나 기울기의 부호에 따라 차이가 생길 수 있다. 보정 전후 Brier와 경보 변화를 각각 확인한다. 보정된 CSV의 `alarm`도 **보정 점수와 보정 임계값으로 다시 계산**한다. 원본에서 복사한 경보 열이 새 점수와 달라질 수 있는 경우를 방지한다.

In [ ]:
def logit_scores(values):
    # Example: p=0.5 -> logit=0; p=0 or 1 is clipped before taking logarithms.
    # Example: p=0 becomes 0.000001, yielding a finite logit near -13.82.
    clipped = np.clip(np.asarray(values), 1e-6, 1 - 1e-6)
    logits = np.log(clipped / (1 - clipped))
    # LogisticRegression expects (windows, features): one logit feature here.
    return logits.reshape(-1, 1)

In [ ]:
example_probabilities = np.array([0.0, 0.5, 1.0])
example_logits = logit_scores(example_probabilities)
print("probabilities =", example_probabilities.tolist())
print("logit input shape =", example_logits.shape)  # (3, 1)
print("logit values =", example_logits[:, 0].tolist())

In [ ]:
calibration_ready = False
calibrated_predictions = pd.DataFrame(columns=PREDICTION_COLUMNS + ["alarm"])
calibration_summary = {}
if ready:
    try:
        # The calibrator sees validation labels only; test labels are never fitted.
        calibrator = LogisticRegression(random_state=CALIBRATION_SEED)
        calibrator.fit(logit_scores(validation["score"]), validation["target"])
        # Column 1 is calibrated P(abnormal), not the probability of a correct alarm.
        validation_scores = calibrator.predict_proba(logit_scores(validation["score"]))[:, 1]
        calibrated_predictions = operating_predictions.copy()
        calibrated_predictions["score"] = calibrator.predict_proba(logit_scores(testing["score"]))[:, 1]
        # The calibrated threshold also comes from validation negatives only.
        calibrated_predictions["threshold"] = threshold90(validation["target"], validation_scores)
        # Recompute alarms; a copied column may still contain the raw decisions.
        calibrated_predictions["alarm"] = (
            calibrated_predictions["score"] >= calibrated_predictions["threshold"]
        )
        # A positive slope is increasing; a negative slope can reverse score order.
        calibration_summary = dict(
            slope=float(calibrator.coef_[0, 0]), intercept=float(calibrator.intercept_[0]),
            threshold=float(calibrated_predictions["threshold"].iloc[0]),
            changed_alarms=int((calibrated_predictions["alarm"] != operating_predictions["alarm"]).sum()),
        )
        calibration_ready = True
    except (ValueError, KeyError) as error:
        calibrated_predictions = pd.DataFrame(columns=PREDICTION_COLUMNS + ["alarm"])
        issues.append(dict(stage="calibration", job="validation_fit", reason=str(error)))

In [ ]:
calibration_difference = pd.DataFrame(columns=METRIC_COLUMNS)
if calibration_ready:
    # Difference = calibrated minus raw. Negative Brier favors calibration.
    calibration_difference = metric_intervals(calibrated_predictions, operating_predictions)
    print("calibration slope =", calibration_summary["slope"])
    print("calibrated threshold =", calibration_summary["threshold"])
    print("changed test alarms =", calibration_summary["changed_alarms"])
    display(calibration_difference)

In [ ]:
if calibration_ready:
    figure, axis = plt.subplots(figsize=(5, 4))
    for label, prediction in (("raw", operating_predictions), ("calibrated", calibrated_predictions)):
        # Ten fixed probability bins; observed=True omits bins with no test windows.
        bins = pd.cut(prediction["score"], np.linspace(0, 1, 11), include_lowest=True)
        # Example: scores [0.11, 0.19], labels [0, 1] -> mean prediction 0.15, outcome 0.5.
        observed = prediction.groupby(bins, observed=True).agg(
            prediction=("score", "mean"), outcome=("target", "mean"), count=("target", "size"),
        )
        axis.plot(observed["prediction"], observed["outcome"], "o-", label=label)
    # The diagonal marks predicted probability = observed abnormal fraction.
    axis.plot([0, 1], [0, 1], "k--")
    axis.set(xlabel="Mean predicted probability", ylabel="Observed positive fraction")
    axis.legend()
    figure.tight_layout()
    plt.show()

In [ ]:
if ready:
    figure, axis = plt.subplots(figsize=(5, 4))
    for target in (0, 1):
        scores = operating_predictions.loc[operating_predictions["target"].eq(target), "score"]
        axis.hist(scores, bins=25, alpha=0.5, label=f"label {target}")
    # This vertical line is the raw threshold frozen from validation negatives.
    axis.axvline(threshold, color="k", linestyle="--")
    axis.set(xlabel="Raw score", ylabel="Windows")
    axis.legend()
    figure.tight_layout()
    plt.show()

## 6. 고정한 경보 규칙으로 TP/TN/FP/FN 사례를 고른다

각 분류에서 최대 두 구간을 선택한다. TP·FP는 높은 점수부터, TN·FN은 낮은 점수부터이며 동점은 `row_id`로 결정한다. 선택 사례와 사례별 SHAP는 내부 결과 파일에만 저장하며, 노트북에는 개별 행·파형을 출력하지 않는다.

사례 파형 입력의 존재 여부는 확인하되 그림은 만들지 않는다. SHAP에는 별도로 평활 특징 표를 사용한다.

In [ ]:
def select_cases(prediction):
    parts = []
    for name, target, alarm in (("TP", 1, True), ("TN", 0, False), ("FP", 0, True), ("FN", 1, False)):
        subset = prediction.loc[prediction["target"].eq(target) & prediction["alarm"].eq(alarm)]
        # Example: two FP rows with scores 0.9 and 0.7 -> inspect 0.9 first.
        # Same score -> row_id ascending; TN/FN use score ascending.
        chosen = subset.sort_values(["score", "row_id"], ascending=[not alarm, True]).head(2)
        parts.append(chosen.assign(case_type=name))
    # Group order stays TP, TN, FP, FN; an empty group contributes no rows.
    return pd.concat(parts, ignore_index=True)

In [ ]:
cases = pd.DataFrame(columns=PREDICTION_COLUMNS + ["alarm", "case_type"])
if ready:
    cases = select_cases(operating_predictions)
    print("selected case counts =", cases["case_type"].value_counts().to_dict())

In [ ]:
waves_ready = False
if not cases.empty:
    try:
        # Read only selected signal_index rows without loading the entire array into RAM.
        waves = np.load(RESULTS / "cnn_signals.npy", mmap_mode="r")
        if waves.ndim != 3 or waves.shape[1:] != (2, 150):
            raise ValueError("사례 파형은 (구간 수, 2, 150) 배열이어야 합니다")
        indices = cases["signal_index"].to_numpy()
        if not np.isfinite(indices).all() or not np.equal(indices, indices.astype(int)).all():
            raise ValueError("사례 신호 인덱스가 정수가 아닙니다")
        if np.any(indices < 0) or np.any(indices >= len(waves)):
            raise ValueError("사례 신호 인덱스가 배열 범위를 벗어납니다")
        waves_ready = True
        print("wave array shape =", waves.shape)
    except (OSError, ValueError) as error:
        issues.append(dict(stage="cases", job="waveforms", reason=str(error)))

## 7. 선택된 같은 모델의 사례별 출력 기여를 읽는다

저장된 Cat28과 사례의 평활 특징 28개로 SHAP를 구한다. 반환 배열은 구간마다 29개 값이며, 앞의 28개가 특징 기여, 마지막 값이 기준 출력이다. 기준 출력과 28개 기여를 더하면 모델의 raw logit 출력이 된다. 각 사례에서 절댓값이 큰 기여 다섯 개를 저장한다. 기여의 부호는 이 모델의 이상 logit을 높이거나 낮추는 방향이다.

In [ ]:
# Match the feature order used to fit the saved Cat28 model in 04.
CAT28 = [
    "fhr_mean", "fhr_min", "fhr_max", "fhr_sd", "stv", "brady_frac", "tachy_frac",
    "n_decel", "decel_max_depth", "decel_time_frac", "n_accel",
    "toco_mean", "toco_max", "toco_sd", "n_contractions",
    "ft_corr0", "ft_corr_min", "ft_corr_min_lag_s", "figo_baseline", "figo_baseline_var",
    "n_early_decel", "n_late_decel", "n_variable_decel",
    "n_severe_decel", "n_prolonged_decel", "hist_width", "hist_median", "hist_mode",
]

In [ ]:
model_ready = False
feature_table = pd.DataFrame()
if ready:
    try:
        from catboost import CatBoostClassifier, Pool
        feature_table = pd.read_csv(RESULTS / "features.csv", dtype=ID_TYPES)
        if not feature_table["row_id"].is_unique:
            raise ValueError("SHAP 특징 표의 구간 ID가 중복됩니다")
        feature_table[CAT28]
        selection = json.loads((RESULTS / "model_selection.json").read_text(encoding="utf-8"))
        # Example: selected Cat28 seed 43 -> models/cat28_43.cbm.
        selected_seed = selection["cat28"]["seed"]
        model = CatBoostClassifier()
        model.load_model(str(RESULTS / "models" / f"cat28_{selected_seed}.cbm"))
        model_ready = True
        print("selected Cat28 model seed =", selected_seed)
    except Exception as error:
        # Missing model/feature files affect explanations, not operating metrics.
        issues.append(dict(stage="SHAP", job="model_inputs", reason=str(error)))

In [ ]:
case_contributions = pd.DataFrame(columns=["row_id", "case", "feature", "value", "contribution"])
if model_ready and not cases.empty:
    try:
        # Preserve case order when connecting predictions to feature rows.
        case_inputs = feature_table.set_index("row_id").loc[cases["row_id"], CAT28]
        # Example: 8 selected windows -> SHAP shape (8, 29): 28 features + base value.
        case_shap = model.get_feature_importance(Pool(case_inputs), type="ShapValues")
        contribution_rows = []
        for position, row in enumerate(cases.itertuples()):
            # Example: [-0.2, +0.8, -0.5] ranks by magnitude as [1, 2, 0].
            # The last SHAP column is the base value, not a 29th feature.
            # The top five alone do not reconstruct the full model output.
            order = np.argsort(-np.abs(case_shap[position, :-1]))[:5]
            for feature_index in order:
                contribution_rows.append(dict(
                    row_id=row.row_id, case=row.case_type, feature=CAT28[feature_index],
                    value=case_inputs.iloc[position, feature_index],
                    contribution=case_shap[position, feature_index],
                ))
        case_contributions = pd.DataFrame(contribution_rows)
        print("case SHAP shape =", case_shap.shape)
    except Exception as error:
        issues.append(dict(stage="SHAP", job="cases", reason=str(error)))

## 8. 훈련 구간의 평균 절대 SHAP를 계산한다

훈련 집합에서 최대 512개 구간을 01의 `split_seed`로 뽑는다. 이 표본의 SHAP 절댓값을 특징별로 평균낸다. 표본 추출 seed와 Cat28 모델 선택 seed는 역할이 다르다. 이 순위는 모델 출력 기여 크기이며, 05에서 특징군을 제거하고 **다시 학습**했을 때의 성능 차이와 같은 양이 아니다.

In [ ]:
global_importance = pd.DataFrame(columns=["feature", "mean_abs_shap"])
if model_ready:
    try:
        training = feature_table.loc[feature_table["split"].eq("train")]
        # Example: 800 train windows -> sample 512; 100 train windows -> use all 100.
        # Sample seed comes from 01, independently of the selected model seed.
        sample = training.sample(n=min(512, len(training)), random_state=protocol["split_seed"])
        global_shap = model.get_feature_importance(Pool(sample[CAT28]), type="ShapValues")
        raw_logits = model.predict(sample[CAT28], prediction_type="RawFormulaVal")
        # Example: base=-1.0, contributions=[+0.3, -0.2] -> raw logit=-0.9.
        np.testing.assert_allclose(global_shap.sum(axis=1), raw_logits, atol=1e-8)
        # Example: contributions [-0.2, +0.4] -> mean absolute SHAP=0.3.
        # Exclude the base-value column before averaging over sampled windows.
        global_importance = pd.DataFrame(dict(
            feature=CAT28, mean_abs_shap=np.abs(global_shap[:, :-1]).mean(axis=0),
        ))
        print("training SHAP sample rows =", len(sample))
        print("training SHAP shape =", global_shap.shape)
    except Exception as error:
        issues.append(dict(stage="SHAP", job="global", reason=str(error)))

In [ ]:
if not global_importance.empty:
    # Sort only for display; the CSV retains the original Cat28 feature order.
    global_importance.sort_values("mean_abs_shap").set_index("feature").plot.barh(
        figsize=(8, 8), legend=False,
    )
    plt.title("Training sample: mean absolute SHAP on logit scale")
    plt.tight_layout()
    plt.show()
    display(global_importance.sort_values("mean_abs_shap", ascending=False))

## 9. 운영 지표·보정·사례를 저장한다

| 파일 | 내용 |
| --- | --- |
| `06_threshold.json` | 검증 음성으로 정한 원점수 임계값 |
| `pred_cat28_operating.csv` | 원점수·고정 임계값·경보가 있는 시험 구간 |
| `06_operating_metrics.csv` | 운영 지표 8개와 산모 95% CI |
| `pred_cat28_calibrated.csv` | 보정 점수·보정 임계값·다시 계산한 경보 |
| `06_calibration_difference.csv` | 보정 후−보정 전 지표 차이와 짝지은 CI |
| `06_cases.csv` | 원래 경보 규칙의 TP/TN/FP/FN 선택 사례 |
| `06_case_contributions.csv` | 사례별 상위 다섯 특징의 값·SHAP 기여 |
| `06_global_shap.csv` | 훈련 표본의 특징별 평균 절대 SHAP |
| `06_issues.csv`, `06_status.json` | 보류한 단계·이유와 완료 상태 |

결과 파일명과 지표·열 순서를 유지한다. 산출하지 못한 표도 빈 스키마로 갱신해 이전 성공 결과를 남기지 않는다. 04의 원래 예측·모델은 입력이며, 운영 경보와 확률 보정 결과는 별도 파일에 저장한다.

In [ ]:
RESULTS.mkdir(parents=True, exist_ok=True)
# Raw and calibrated outputs are separate; each saved alarm matches its own score/threshold.
operating_predictions.to_csv(RESULTS / "pred_cat28_operating.csv", index=False)
performance.to_csv(RESULTS / "06_operating_metrics.csv", index=False)
calibrated_predictions.to_csv(RESULTS / "pred_cat28_calibrated.csv", index=False)
calibration_difference.to_csv(RESULTS / "06_calibration_difference.csv", index=False)
cases.to_csv(RESULTS / "06_cases.csv", index=False)
case_contributions.to_csv(RESULTS / "06_case_contributions.csv", index=False)
global_importance.to_csv(RESULTS / "06_global_shap.csv", index=False)

# A missing threshold is JSON null, never a leftover threshold from an older run.
threshold_info = dict(threshold=threshold, source="validation_negatives")
(RESULTS / "06_threshold.json").write_text(
    json.dumps(threshold_info, ensure_ascii=False, indent=2), encoding="utf-8",
)

In [ ]:
issue_table = pd.DataFrame(issues, columns=["stage", "job", "reason"])
issue_table.to_csv(RESULTS / "06_issues.csv", index=False)
# Operating metrics can succeed even when waveform or SHAP inputs are unavailable.
if performance.empty:
    status_name = "unavailable"
elif issue_table.empty:
    status_name = "complete"
else:
    status_name = "partial"
status = dict(
    status=status_name, operating_metrics_complete=not performance.empty,
    calibration_complete=not calibration_difference.empty, calibration=calibration_summary,
    cases=len(cases), waveforms_complete=waves_ready,
    case_contributions=len(case_contributions), global_features=len(global_importance),
    input_error=input_error, bootstrap=N_BOOTSTRAP, bootstrap_seed=BOOTSTRAP_SEED,
)
(RESULTS / "06_status.json").write_text(
    json.dumps(status, ensure_ascii=False, indent=2), encoding="utf-8",
)
print("status =", status_name)
print("operating metric rows =", len(performance))
print("cases =", len(cases), "global SHAP features =", len(global_importance))
print("issues =", len(issue_table))
if not issue_table.empty:
    print("issue count =", len(issue_table))

## 결과를 해석하는 범위

검증 특이도가 90% 이상이어도 다른 시험 산모나 사이트에서 같은 특이도를 보장하지 않는다. 정상 시간당 경보는 유지된 5분 창의 계산값이며 실제 연속 모니터링의 경보 episode 부담과 다르다. 확률 보정과 SHAP를 근거로 임상 효용이나 전문의 대체 가능성을 바로 결론내리지 않는다.

사례와 SHAP는 선택된 모델의 내부 오류·출력 기여를 살펴보는 자료다. 생리적 인과 관계나 전문의의 실제 판단 과정을 설명하지 않는다. 산모 bootstrap은 고정된 모델·임계값·보정기의 시험 예측을 재표집하므로 재학습과 선택 전체의 변동을 포함하지 않는다.

### 반출용 제외·오류 집계

개별 ID·경로·오류 전문을 제외한 분류별 건수다.

In [ ]:
show_issue_counts(issue_table["reason"])